In [2]:
from typing import Literal, Dict, List, TypedDict

import shared_libraries.simulation as sim
import shared_libraries._simulation_utils as _simutils

import pandas as pd
import numpy as np
import pickle

import statistics
import random
import os
from pathlib import Path

import matplotlib.pyplot as plt

In [3]:
with open("ig_models/models.pickle", "rb") as file:
    models = pickle.load(file)

In [4]:
circuit_lap_counts = pd.read_csv("./ig_data/circuit_lap_counts.csv", index_col=0)
circuit_lap_counts

,LapCount
Sakhir,57
Jeddah,50
Melbourne,58
Miami,57
Catalunya,66
Monte Carlo,78
Baku,51
Montreal,70
Silverstone,52
Paul Ricard,53


In [5]:
compound_mappings = sim.prepare_example_compound_mappings()
weathers = sim.prepare_example_weather_data()
simulation_results = sim.prepare_simulation(
    circuit_lap_counts.index,
    compound_mappings,
    weathers,
    circuit_lap_counts
)

In [ ]:

total_circuits = len(simulation_results["FullStrategyData"])
i = 1
print()
for circuit, strategy_data_by_conditions in simulation_results["FullStrategyData"].items():
    print(f"\rProcessing circuit {i}/{total_circuits}", end="")
    i += 1
    model = models[circuit]["XGBRegressor"].best_estimator_
    for conditions_id, strategy_data in strategy_data_by_conditions.items():

        strategy_data_post_evaluation = _simutils.evaluate_strategies(strategy_data, model)
        
        laps = strategy_data_post_evaluation["Laps"]
        strategies = strategy_data_post_evaluation["Strategies"]
        conditions = simulation_results["WeatherAndMappingCombinations"].loc[conditions_id]
        mapping_name: str = conditions["WeatherName"] # type: ignore
        weather_name: str = conditions["CompoundMappingName"] # type: ignore

        best_strat_id = strategies.sort_values(by="MeanZScore", axis="index", ascending=False).iloc[0]["Id"]
        strat = strategies.loc[best_strat_id, "Strategy"]

        best_strat_laps = laps[laps["StrategyId"] == best_strat_id]
        race_length = best_strat_laps.shape[0]
        stints = []
        current_compound = strat.initial_compound
        current_lap = 1
        for pit_lap, compound in strat.stops.items():
            start = current_lap
            while current_lap < pit_lap:
                current_lap += 1
            stints.append({"start": start, "end": current_lap, "compound": current_compound})
            current_lap += 1
            current_compound = compound
        stints.append({"start": current_lap, "end": race_length, "compound": current_compound})
        
        fig, (chart_ax, stint_ax) = plt.subplots(2, 1, height_ratios=(2, 1), sharex=True)
        chart_ax.grid(axis="x")
        chart_ax.plot(best_strat_laps["LapNumber"], best_strat_laps["LapTimeZScore"])
        chart_ax.set_xticks(range(1, race_length + 1, 3))
        chart_ax.set_xlabel("Lap number")

        first_stop = True
        for stop in strat.stops:
            chart_ax.axvline(stop, color="red", linestyle=":", linewidth=2, label="pit stop" if first_stop else None)
            first_stop = False

        compound_colors = {"SOFT": "red", "MEDIUM": "yellow", "HARD": "white"}
        for stint in stints:
            compound = stint["compound"]
            stint_length = stint["end"] - stint["start"] + 1

            color = compound_colors.get(
                stint["compound"], "grey"
            )

            stint_ax.margins(0, 0)
            stint_ax.barh(0, width=stint_length, height=1, left=stint["start"], color=color, edgecolor="black")
            stint_ax.text(np.mean([stint["start"], stint["end"]]), 0, compound, va="center", ha="center", fontweight="bold")


        stint_ax.set_xticks(range(1, race_length + 1, 3))
        stint_ax.set_yticks([])

        path = Path(f"./figures/simulation_results/{circuit}/{weather_name.replace(' ', '_')}__{mapping_name.replace(' ', '_')}")

        os.makedirs(path.parent, exist_ok=True)
        fig.savefig(path)
        plt.close(fig)
print()



Processing circuit 12/22